In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path(sys.prefix).parent
%cd {PROJECT_ROOT}

/home/younes/younes/Projects/Python/barid_internship


In [24]:
import polars as pl

import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.graphics.tsaplots import plot_acf
import calendar
from fpppy.utils import plot_series, plot_series_stacked, plot_diagnostics
from scipy.stats import pearsonr
from plotly import express as px
import altair as alt
from pathlib import Path
from lxml import html
from itertools import chain
import lib

import importlib
import offline_historique

importlib.reload(offline_historique)
from offline_historique import parse_historique_from_folder  # noqa: E402

cfg = pl.Config()
cfg.set_tbl_width_chars(10000)
cfg.set_fmt_str_lengths(100)
cfg.set_tbl_cols(-1)
cfg.set_tbl_rows(10)

polars.config.Config

In [106]:
fields = pl.read_parquet("data/cab_dfs/fields.parquet").sort("Date_depot").drop_nulls("Poids_global_en_KG")
operations = pl.read_parquet("data/cab_dfs/operations.parquet")
delivery = pl.read_parquet("data/cab_dfs/delivery.parquet")
services = pl.read_parquet("data/cab_dfs/services.parquet")

In [20]:
print(fields.head())

shape: (5, 21)
┌───────────────┬──────────┬────────────┬──────────┬────────────────┬────────┬─────────┬──────┬────────────────────┬───────────────────────────────────┬────────────────────────┬────────────────────┬────────────────────────┬────────────────────────────────────────────────────┬──────────────┬──────────────────────┬──────────────────────┬──────────┬─────────┬─────────┬────────────────────┐
│ Cab           ┆ Id       ┆ Date_depot ┆ Type_cab ┆ Dernier_statut ┆ Regime ┆ Contrat ┆ Etat ┆ Poids_global_en_KG ┆ Centre_Agence_depot               ┆ Destination            ┆ Client             ┆ Produit_Niveau_Service ┆ Mode_paiement                                      ┆ Taxe_DTQ_Dhs ┆ Canal_de_livraison_1 ┆ Canal_de_livraison_2 ┆ Longueur ┆ Hauteur ┆ Largeur ┆ Poids_Volumetrique │
│ ---           ┆ ---      ┆ ---        ┆ ---      ┆ ---            ┆ ---    ┆ ---     ┆ ---  ┆ ---                ┆ ---                               ┆ ---                    ┆ ---                ┆ ---   

In [ ]:
def prepare_data(df: pl.DataFrame, period: str):
    return (
        df.drop_nulls("Poids_global_en_KG")
        .group_by_dynamic(
            index_column="Date_depot",
            every=period,
        )
        .agg(
            pl.col("Poids_global_en_KG").sum(),
            len=pl.len(),
            mean_weight=pl.col("Poids_global_en_KG").sum() / pl.len(),
        )
        .pipe(
            lib.complete_grid,
            dimensions={
                "Date_depot": lambda d: pl.date_range(
                    start=d["Date_depot"].min(),  # pyright: ignore[reportArgumentType]
                    end=d["Date_depot"].max(),  # pyright: ignore[reportArgumentType]
                    interval=period,
                    eager=True,
                ),  # type: ignore
            },
            fill_value=0,
        )
        .filter(pl.col("Date_depot").ge(pl.date(2025, 1, 1)))
    )


print(prepare_data(fields, "1d"))

shape: (493, 4)
┌────────────┬────────────────────┬─────┬─────────────┐
│ Date_depot ┆ Poids_global_en_KG ┆ len ┆ mean_weight │
│ ---        ┆ ---                ┆ --- ┆ ---         │
│ date       ┆ f64                ┆ u32 ┆ f64         │
╞════════════╪════════════════════╪═════╪═════════════╡
│ 2025-01-01 ┆ 0.0                ┆ 0   ┆ 0.0         │
│ 2025-01-02 ┆ 1257.615           ┆ 415 ┆ 3.030398    │
│ 2025-01-03 ┆ 1010.92            ┆ 327 ┆ 3.091498    │
│ 2025-01-04 ┆ 15.181             ┆ 12  ┆ 1.265083    │
│ 2025-01-05 ┆ 19.8               ┆ 5   ┆ 3.96        │
│ …          ┆ …                  ┆ …   ┆ …           │
│ 2026-05-04 ┆ 0.0                ┆ 0   ┆ 0.0         │
│ 2026-05-05 ┆ 0.1                ┆ 1   ┆ 0.1         │
│ 2026-05-06 ┆ 0.0                ┆ 0   ┆ 0.0         │
│ 2026-05-07 ┆ 0.0                ┆ 0   ┆ 0.0         │
│ 2026-05-08 ┆ 0.05               ┆ 1   ┆ 0.05        │
└────────────┴────────────────────┴─────┴─────────────┘


In [130]:
chart = (
    alt.Chart(
        fields.pipe(prepare_data, "7d"),
    )
    .mark_line()
    .encode(x="Date_depot:T", y="mean_weight:Q")
    .properties(width=1100)
)
chart

alt.Chart(...)

In [ ]:
df = (
    fields.sort("Poids_global_en_KG")
    .with_columns(
        pl.col("Poids_global_en_KG")
        .cut(
            [float(x) for x in range(int(fields["Poids_global_en_KG"].max()))],
            left_closed=True,
        )
        .alias("bucket")
    )
    .group_by("bucket", maintain_order=True)
    .agg(
        pl.col("Poids_global_en_KG").mean().alias("Poids_global_en_KG"),
        pl.col("Poids_global_en_KG").sum().alias("weight_sum"),
    )
    .with_columns(cum_sum=pl.col("weight_sum").cum_sum(reverse=True))
)

base = alt.Chart(df).properties(width=1100).mark_line(point=True)

sum_chart = base.encode(
    x=("bucket"),
    y="weight_sum",
    tooltip=[
        "bucket",
        "weight_sum",
    ],
)

cum_sum_chart = base.encode(
    x="bucket",
    y="cum_sum",
    tooltip=[
        "bucket",
        "cum_sum",
    ],
)
chart = sum_chart & cum_sum_chart

chart.resolve_legend()

alt.VConcatChart(...)